In [1]:
import pandas as pd
import numpy as np
import os

city = "amsterdam"
folder = f"/mnt/common-hdd/raw-sources/twitter-data/raw-data/{city}"
tweet_folder = os.path.join(folder, "tweets")

filelist = os.listdir(tweet_folder)
test_file = os.path.join(tweet_folder,filelist[0])

print(test_file)

/mnt/common-hdd/raw-sources/twitter-data/raw-data/amsterdam/tweets/amsterdam_2018-12-02_2018-12-03.csv


## Tweet table

In [4]:
tweet = pd.read_csv(test_file)
# shape
print(tweet.shape)
m = tweet["geo_coo_coordinates"] == "Amsterdam, Nederland"
# print % of m==True
print(f"Percentage of tweets from Amsterdam: {m.mean()*100:.2f}%, number of tweets: {m.sum()} out of {len(tweet)}")
tweet[m].head(5).T

(3750, 39)
Percentage of tweets from Amsterdam: 0.24%, number of tweets: 9 out of 3750


,2608,2614,2701,2733,2843
attachments,------------Leven tot nu toe.,------------Leven tot nu toe.,------------Leven tot nu toe.,------------Leven tot nu toe.,------------Leven tot nu toe.
author_created_at,NaN,NaN,NaN,NaN,NaN
author_description,822734506641735680,822734506641735680,822734506641735680,822734506641735680,822734506641735680
author_entities,Eemnes,Eemnes,Eemnes,Eemnes,Eemnes
author_id,Cees,Cees,Cees,Cees,Cees
author_location,NaN,NaN,NaN,NaN,NaN
author_name,428,428,428,428,428
author_pinned_tweet_id,420.0,420.0,420.0,420.0,420.0
author_pm_followers_count,1.0,1.0,1.0,1.0,1.0
author_pm_following_count,12135.0,12135.0,12135.0,12135.0,12135.0


In [5]:
"""
    tweet_id bigint NOT NULL,
    user_id bigint NOT NULL,
    created_at timestamp without time zone NOT NULL,
    place_id VARCHAR(50),
    lat FLOAT,
    lon FLOAT
"""

'\n    tweet_id bigint NOT NULL,\n    user_id bigint NOT NULL,\n    created_at timestamp without time zone NOT NULL,\n    place_id VARCHAR(50),\n    lat FLOAT,\n    lon FLOAT\n'

In [39]:
column_rename = {
    "id": "tweet_id",
    "author_id": "user_id",
    "created_at": "created_at",
    "geo_place_id": "place_id",
    "lat": "lat",
    "lon": "lon"
}

In [40]:
tweet.rename(columns=column_rename, inplace=True)

In [23]:
from ast import literal_eval

In [36]:
def extract_lat(coord_col):
    if type(coord_col) is not str:
        return None
    else:
        try:
            return literal_eval(coord_col)[1]
        except:
            # 8 rows where shifted by 2 right from the first column, so place_name is in geo_coo_coordinates
            # print(coord_col)
            return None
    
def extract_lon(coord_col):
    if type(coord_col) is not str:
        return None
    else:
        try:
            return literal_eval(coord_col)[0]
        except:
            # 8 rows where shifted by 2 right from the first column, so place_name is in geo_coo_coordinates
            # print(coord_col)
            return None

In [ ]:
tweet["lat"] = tweet["geo_coo_coordinates"].map(extract_lat)
tweet["lon"] = tweet["geo_coo_coordinates"].map(extract_lon)

In [19]:
tweet["lat"].dropna()

Series([], Name: lat, dtype: object)

In [ ]:
tweet[['tweet_id', 'user_id', 'created_at', 'place_id', 'lat', 'lon']].head(2).T
# additional column: city

,0,1
tweet_id,1069378436857307136,1069377759678513152
user_id,844596662265921536,23675631
created_at,2018-12-02 23:51:02+00:00,2018-12-02 23:48:20+00:00
place_id,591e44cbdb12426e,cd003ebe3a96fcc6
lat,NaN,NaN
lon,NaN,NaN


## Network table 1 - followers

Users who are included:
* at least 10 tweets
* at least 10 different days in the data
* at least 1 months max - min time
* good enough following / follower ratio

**Aim** : follower table
* city
* user_id1_source
* user_id2_target


`/mnt/common-hdd/raw-sources/twitter-data/data` itt talaltunk follower mappat es follower_edge.csv fajlt

1. A `follower_edge.csv`-ben forditott sorrendben vannak lejegyezve az oszlopok. -> following = user_id1_source, follower = user_id2_target
2. Melyik varosbol van meg a follower halo (valoszinuleg Amszterdam, Portland)?
3. Pontosan mi volt a bemeneti userlista a follower halo gyujtesehez? (Eszter - koncepcio, Bence - source fajlok)
4. Mi a `{user_id}_follower.csv` fajl tartalma?

In [6]:
follower = pd.read_csv("/mnt/common-hdd/raw-sources/twitter-data/data/follower_edge.csv")

## Network table 2 - mentions, retweets, replies

**aim** : conversation networks
* city
* tweet_id
* created_at
* user_id1_source (author of tweet)
* user_id2_interaction (row['entities']['mentions']['id'] - can be multiple, `in_reply_to_user_id`)
* type (m/rt/re)

Hol vannak a retweetek?

## Place database

* place_id
* name
* full_name
* country
* place_type
* bounding_box.coordinates -> lon_min, lon_max, lat_min, lat_max